# SOB4ES — 03. Unión y Estandarización Final
### CRISP-ML(Q) · Fase 2: Ingeniería de Datos

Este notebook combina las salidas de los dos notebooks anteriores en un único dataset listo para modelado.

| Paso | Descripción |
|------|-------------|
| **1** | Configuración y rutas |
| **2** | Cargar salidas de notebooks 01 y 02 |
| **3** | Corregir nombres de columna rotos (snake_case sobre acrónimos) |
| **4** | Unir datos locales + online |
| **5** | Imputar NaN de las variables online |
| **6** | Escalar variables online con el mismo scaler del notebook 01 |
| **7** | Exportar dataset final combinado |

**Nota: **La x en vx se refiere a la versión/iteración del archivo, esto se aplica a todos los outputs empleados.

**Entradas**: `sob4es_clean_vx.csv`, `sob4es_model_ready_vx.csv`, `online_features.csv`, `scaler.pkl`, `label_encoders.pkl`  
**Salidas**: `sob4es_final_clean.csv`, `sob4es_final_model_ready.csv`

## 0.- Configuración

In [1]:
import os, sys, warnings, re
import numpy  as np
import pandas as pd
import joblib
from sklearn.preprocessing import StandardScaler, LabelEncoder

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', '{:.4f}'.format)

OUT_DIR = 'output/'

os.makedirs(OUT_DIR, exist_ok=True)

# Rutas de entrada
CLEAN_PATH   = OUT_DIR + 'sob4es_clean_v4.csv'
MODEL_PATH   = OUT_DIR + 'sob4es_model_ready_v4.csv'
ONLINE_PATH  = OUT_DIR + 'online_features.csv'
SCALER_PATH  = OUT_DIR + 'scaler.pkl'
ENCODER_PATH = OUT_DIR + 'label_encoders.pkl'

for nombre, ruta in [
    ('CLEAN_PATH',   CLEAN_PATH),
    ('MODEL_PATH',   MODEL_PATH),
    ('ONLINE_PATH',  ONLINE_PATH),
    ('SCALER_PATH',  SCALER_PATH),
    ('ENCODER_PATH', ENCODER_PATH),
]:
    existe = os.path.exists(ruta)
    print(f'  {nombre:15s}: {ruta}  {"Encontrado..." if existe else "[ERROR] Archivo no encontrado, por favor revisa las rutas..."}')

  CLEAN_PATH     : output/sob4es_clean_v4.csv  Encontrado...
  MODEL_PATH     : output/sob4es_model_ready_v4.csv  Encontrado...
  ONLINE_PATH    : output/online_features.csv  Encontrado...
  SCALER_PATH    : output/scaler.pkl  Encontrado...
  ENCODER_PATH   : output/label_encoders.pkl  Encontrado...


## 1.- Cargar Salidas de los Notebooks Anteriores

In [2]:
df_clean  = pd.read_csv(CLEAN_PATH)
df_model  = pd.read_csv(MODEL_PATH)
df_online = pd.read_csv(ONLINE_PATH)

print(f'clean      : {df_clean.shape}')
print(f'model_ready: {df_model.shape}')
print(f'online     : {df_online.shape}')
print()
print('Online columns:', list(df_online.columns))

clean      : (427, 115)
model_ready: (427, 102)
online     : (427, 8)

Online columns: ['SITE_ID', 'gee_temp_media_C', 'gee_humedad_rel_pct', 'gee_ndvi_verano', 'cds_precip_mm_mes', 'dem_elevacion_m', 'dem_pendiente_deg', 'dem_orientacion_deg']


## 2.- Corregir Nombres de Columna Rotos

La función `estandarizar_nombre()` del notebook 01 aplicó snake_case sobre columnas
que ya eran acrónimos en mayúsculas (ej. `SITE_ID`, `BACTERIA_SHANNON`, `CN`).
El resultado: cada letra separada por guiones bajos (`s_i_t_e_i_d`, `b_a_c_t_e_r_i_a...`).

Se corrigen aquí con un mapeo explícito antes de hacer cualquier unión.

In [ ]:
# Mapeo de nombres rotos → nombres correctos
# Columnas afectadas: acrónimos y nombres que el CamelCase splitter separó letra a letra
RENAME_MAP = {
    # ID principal (clave de unión)
    'sob4es_s_i_t_e_i_d'                           : 'SITE_ID',

    # Shannon microbianos — acrónimos en mayúscula fragmentados
    'sob4es_b_a_c_t_e_r_i_a_s_h_a_n_n_o_n'        : 'sob4es_bacteria_shannon',
    'sob4es_f_u_n_g_i_s_h_a_n_n_o_n'              : 'sob4es_fungi_shannon',
    'sob4es_e_u_k_a_r_y_o_t_e_s_s_h_a_n_n_o_n'   : 'sob4es_eukaryotes_shannon',
    'sob4es_n_e_m_a_t_o_d_e_shannon'               : 'sob4es_nematode_shannon',

    # EU rasters — acrónimos fragmentados
    'eu_c_n_ratio'                                  : 'eu_cn_ratio',
    'eu_p_h'                                        : 'eu_ph',

    # _was_missing derivadas de columnas rotas
    'sob4es_b_a_c_t_e_r_i_a_s_h_a_n_n_o_n_was_missing' : 'sob4es_bacteria_shannon_was_missing',
    'sob4es_f_u_n_g_i_s_h_a_n_n_o_n_was_missing'        : 'sob4es_fungi_shannon_was_missing',
    'sob4es_e_u_k_a_r_y_o_t_e_s_s_h_a_n_n_o_n_was_missing': 'sob4es_eukaryotes_shannon_was_missing',
    'eu_c_n_ratio_was_missing'                      : 'eu_cn_ratio_was_missing',
    'eu_p_h_was_missing'                            : 'eu_ph_was_missing',

    # _z en model_ready
    'sob4es_b_a_c_t_e_r_i_a_s_h_a_n_n_o_n_z'      : 'sob4es_bacteria_shannon_z',
    'sob4es_f_u_n_g_i_s_h_a_n_n_o_n_z'            : 'sob4es_fungi_shannon_z',
    'sob4es_e_u_k_a_r_y_o_t_e_s_s_h_a_n_n_o_n_z' : 'sob4es_eukaryotes_shannon_z',
    'sob4es_n_e_m_a_t_o_d_e_shannon_z'             : 'sob4es_nematode_shannon_z',
    'eu_c_n_ratio_z'                               : 'eu_cn_ratio_z',
    'eu_p_h_z'                                     : 'eu_ph_z',
}

df_clean  = df_clean.rename(columns=RENAME_MAP)
df_model  = df_model.rename(columns=RENAME_MAP)

# Verificar que SITE_ID existe ahora
assert 'SITE_ID' in df_clean.columns, 'SITE_ID no encontrado tras el renombrado'
print(f'SITE_ID en clean  :  ({df_clean["SITE_ID"].nunique()} sitios únicos)')
print(f'SITE_ID en online : {"Mismo tamaño" if "SITE_ID" in df_online.columns else "[ERROR] Esto no debería de pasar, así que si lees esto ten miedo..."}')

# Mostrar columnas que aún tienen el patrón roto (x_y_z)
still_broken = [c for c in df_clean.columns if re.search(r'(?<![a-z])([a-z])_([a-z])_([a-z])', c)]
if still_broken:
    print(f'\nColumnas aún rotas (añadir a RENAME_MAP):')
    for c in still_broken:
        print(f'  {c}')
else:
    print('\nTodos los nombres de columna corregidos...')

SITE_ID en clean  : ✓  (427 sitios únicos)
SITE_ID en online : ✓

Todos los nombres de columna corregidos...


## 3.- Unir Datos Locales + Online

In [4]:
# Unir clean + online por SITE_ID
df_full = df_clean.merge(df_online, on='SITE_ID', how='left', suffixes=('', '_online'))

assert len(df_full) == len(df_clean), f'Pérdida de filas: {len(df_clean)} → {len(df_full)}'
print(f'Clean  : {df_clean.shape}')
print(f'Online : {df_online.shape}')
print(f'Unido  : {df_full.shape}')
print()

# Cobertura de variables online
online_feat_cols = [c for c in df_online.columns if c != 'SITE_ID']
cobertura_online = (
    df_full[online_feat_cols].notna().sum() / len(df_full) * 100
).round(1)
print('Cobertura variables online:')
for col, pct in cobertura_online.items():
    print(f'  {col:30s}  {pct:.1f}%')

Clean  : (427, 115)
Online : (427, 8)
Unido  : (427, 122)

Cobertura variables online:
  gee_temp_media_C                99.8%
  gee_humedad_rel_pct             99.8%
  gee_ndvi_verano                 97.9%
  cds_precip_mm_mes               0.0%
  dem_elevacion_m                 99.1%
  dem_pendiente_deg               99.1%
  dem_orientacion_deg             99.1%


In [5]:
# Añadir SITE_ID al model_ready uniendo por lat/lon
# model_ready no tiene SITE_ID — se recupera desde clean via lat/lon
site_id_lookup = df_clean[['SITE_ID', 'latitude', 'longitude']].copy()
df_model = df_model.merge(site_id_lookup, on=['latitude', 'longitude'], how='left')

# Verificar que todos los sitios tienen SITE_ID
missing_sid = df_model['SITE_ID'].isna().sum()
if missing_sid:
    print(f'[WARN] {missing_sid} filas sin SITE_ID tras el merge lat/lon')
else:
    print(f'SITE_ID recuperado en model_ready:  ({df_model["SITE_ID"].nunique()} sitios)')

SITE_ID recuperado en model_ready:  (427 sitios)


## 4.- Imputar NaN en Variables Online

Misma estrategia que en el notebook 01:
- Variables continuas < 5% NaN: mediana
- Variables continuas 5-50% NaN: columna `_was_missing` + mediana
- \> 50% NaN → eliminar

In [6]:
# Solo aplicar a las columnas online (prefijos gee_, cds_, dem_)
online_num_cols = [c for c in online_feat_cols
                   if df_full[c].dtype in [np.float64, np.float32, np.int64, np.int32]]

miss_pct = df_full[online_num_cols].isnull().mean() * 100

# Eliminar columnas > 50% NaN
drop_online = miss_pct[miss_pct > 50].index.tolist()
if drop_online:
    print(f'Eliminando {len(drop_online)} columnas online con >50% NaN: {drop_online}')
    df_full = df_full.drop(columns=drop_online)
    online_num_cols = [c for c in online_num_cols if c not in drop_online]
    miss_pct = miss_pct.drop(index=drop_online)

# Indicadores _was_missing para columnas 5-50% NaN
cols_indicator = miss_pct[(miss_pct >= 5) & (miss_pct <= 50)].index.tolist()
for col in cols_indicator:
    df_full[col + '_was_missing'] = df_full[col].isna().astype(int)
print(f'Indicadores _was_missing añadidos para variables online: {len(cols_indicator)}')

# Imputar con mediana
medians_online = df_full[online_num_cols].median()
df_full[online_num_cols] = df_full[online_num_cols].fillna(medians_online)

remaining = df_full[online_num_cols].isnull().sum().sum()
print(f'NaN restantes en variables online: {remaining}')
print(f'Shape tras imputación: {df_full.shape}')

Eliminando 1 columnas online con >50% NaN: ['cds_precip_mm_mes']
Indicadores _was_missing añadidos para variables online: 0
NaN restantes en variables online: 0
Shape tras imputación: (427, 121)


## 5.- Escalar Variables Online

Las variables online (`gee_*`, `cds_*`, `dem_*`) no estaban en el dataset cuando se ajustó
el `scaler.pkl` del notebook 01. 

Se ajusta un scaler **nuevo** solo para estas columnas y se guarda por separado para poder aplicarlo en inferencia.

In [7]:
# Columnas online a escalar (excluir _was_missing — son binarias)
online_scale_cols = [
    c for c in online_num_cols
    if not c.endswith('_was_missing')
]

scaler_online = StandardScaler()
scaled_online = scaler_online.fit_transform(df_full[online_scale_cols])
df_scaled_online = pd.DataFrame(
    scaled_online,
    columns=[c + '_z' for c in online_scale_cols],
    index=df_full.index
)

joblib.dump(scaler_online, OUT_DIR + 'scaler_online.pkl')
print(f'Escaladas {len(online_scale_cols)} variables online')
print(f'Scaler guardado en {OUT_DIR}scaler_online.pkl')
print(f'Columnas escaladas: {list(df_scaled_online.columns)}')

Escaladas 6 variables online
Scaler guardado en output/scaler_online.pkl
Columnas escaladas: ['gee_temp_media_C_z', 'gee_humedad_rel_pct_z', 'gee_ndvi_verano_z', 'dem_elevacion_m_z', 'dem_pendiente_deg_z', 'dem_orientacion_deg_z']


## 6.- Ensamblar y Exportar Dataset Final

In [8]:
# 1. Dataset final limpio (legible, escala original)
df_final_clean = df_full.copy()

final_clean_path = OUT_DIR + 'sob4es_final_clean.csv'
df_final_clean.to_csv(final_clean_path, index=False)
print(f'{final_clean_path} done...')
print(f'Tamaño: {df_final_clean.shape[0]} sitios × {df_final_clean.shape[1]} columnas')

output/sob4es_final_clean.csv done...
Tamaño: 427 sitios × 121 columnas


In [9]:
# 2. Dataset final listo para modelo 
# Une model_ready (con _z y _enc del nb01) + _z online + _was_missing online

# Añadir SITE_ID a model_ready (ya recuperado en celda 3)
# Unir model_ready + columnas online escaladas
df_model_final = (
    df_model
    .merge(
        pd.concat([df_full[['SITE_ID']], df_scaled_online], axis=1),
        on='SITE_ID', how='left'
    )
)

# Añadir también los _was_missing de las variables online
online_wm_cols = [c for c in df_full.columns if c.startswith(('gee_', 'cds_', 'dem_'))
                  and c.endswith('_was_missing')]
if online_wm_cols:
    df_model_final = df_model_final.merge(
        df_full[['SITE_ID'] + online_wm_cols],
        on='SITE_ID', how='left'
    )

# Ordenar columnas: IDs → _z → _enc → _was_missing → outlier_flag
id_cols   = ['SITE_ID', 'latitude', 'longitude']
z_cols    = [c for c in df_model_final.columns if c.endswith('_z')]
enc_cols  = [c for c in df_model_final.columns if c.endswith('_enc')]
wm_cols   = [c for c in df_model_final.columns if c.endswith('_was_missing')]
flag_cols = ['outlier_flag'] if 'outlier_flag' in df_model_final.columns else []

final_model_cols = id_cols + z_cols + enc_cols + wm_cols + flag_cols
final_model_cols = [c for c in final_model_cols if c in df_model_final.columns]

df_model_final = df_model_final[final_model_cols]

final_model_path = OUT_DIR + 'sob4es_final_model_ready.csv'
df_model_final.to_csv(final_model_path, index=False)
print(f'{final_model_path} done...')
print(f' Tamaño (REVISAR): {df_model_final.shape[0]} sitios × {df_model_final.shape[1]} columnas')
print(f' Desglose: {len(z_cols)} _z  |  {len(enc_cols)} _enc  |  {len(wm_cols)} _was_missing')

output/sob4es_final_model_ready.csv done...
 Tamaño (REVISAR): 541 sitios × 109 columnas
 Desglose: 65 _z  |  3 _enc  |  37 _was_missing


## 7.- Resumen Final

In [10]:
# Catálogo de columnas
def inferir_fuente(col):
    if col.startswith('gee_'):   return 'GEE (online)'
    if col.startswith('cds_'):   return 'CDS (online)'
    if col.startswith('dem_'):   return 'DEM (online)'
    if col.startswith('eu_'):    return 'Raster EU'
    if col.startswith('sob4es_'): return 'SOB4ES campo'
    if col in ['SITE_ID','latitude','longitude','country',
               'pedoclimatic_region','sampling_date']: return 'Metadatos'
    if col.endswith('_z'):       return 'Escalado (z)'
    if col.endswith('_enc'):     return 'Codificado (enc)'
    if col.endswith('_was_missing'): return 'Indicador NaN'
    return 'Otro'

catalogo = pd.DataFrame({
    'dtype'    : df_final_clean.dtypes,
    'n_nulos'  : df_final_clean.isnull().sum(),
    'n_unicos' : df_final_clean.nunique(),
    'fuente'   : [inferir_fuente(c) for c in df_final_clean.columns],
})

print('--- Columnas por fuente ---')
print(catalogo.groupby('fuente').size().sort_values(ascending=False).rename('n_columnas').to_string())
print()
print(f'--- RESUMEN FINAL ---')
print(f'Sitios                : {df_final_clean.shape[0]}')
print(f'Columnas (clean)      : {df_final_clean.shape[1]}')
print(f'Columnas (model_ready): {df_model_final.shape[1]}')
print(f'NaN totales (clean)   : {df_final_clean.isnull().sum().sum()}')
print(f'NaN totales (model)   : {df_model_final.isnull().sum().sum()}')


--- Columnas por fuente ---
fuente
SOB4ES campo    77
Raster EU       30
Metadatos        6
DEM (online)     3
GEE (online)     3
Otro             2

--- RESUMEN FINAL ---
Sitios                : 427
Columnas (clean)      : 121
Columnas (model_ready): 109
NaN totales (clean)   : 0
NaN totales (model)   : 0
